**`02_set_up_admin_scope`**

Run this once, for whatever geographic scope you're working on:

- three counties spread across three different states
- a handful of municipalities in different countries
- a whole country
- or the whole world.

The script pulls in the `admin_id` referencing data your project's scope actually needs, so downstream scripts know how to store, split, and retrieve data by state, county, town, etc.

`openplaces` ships with a global admin "spine" already: GADM boundaries plus ISO codes, built once (see `05_ingest_global_administrative_units.ipynb` for documentation).

This notebook checks whether a country-specific source is better than GADM for your scope.

If so, it ingests it, restricted to just the `admin_ids` you asked for.

# What is an `admin_id`?

`openplaces` organizes all its data by `admin_id` -- a
hierarchical geographic identifier, joined by `-`, with up to
four levels:

1. country / territory, e.g. `GB`
2. state / department / region, e.g. `GB-EN`
3. county / municipality, e.g. `GB-EN-BA`
4. town / subdivision, e.g. `GB-EN-BA-BA`

Every dataset in `openplaces` -- parcels, buildings,
transactions, imagery -- is keyed to one of these levels, so
getting your admin referencing right, once, makes everything
downstream line up.

# Default admin IDs

In [ ]:
from openplaces import get_admin

# Works everywhere already
get_admin(level=1).sample(5).sort_index()

In [ ]:
# Works everywhere already
get_admin(level=2, all_columns=True).sample(5).sort_index()

# Which countries are already linked to high-quality data?

Before picking your scope, it helps to see which countries already have a recipe that replaces GADM.

In [ ]:
from openplaces.diagnostics import map_admin_recipe_source_coverage

fig, axes = map_admin_recipe_source_coverage(verbose=True)

# Choose your scope

Set `SCOPE` to any number of admin_ids you're working on, at any
depth, mixed freely. This notebook groups them by country
under the hood and ingests only what each entry actually needs.

Leave it as an empty list to stick with the global default only, for
now -- you can always come back and add to it later. Nothing here
touches the global GADM+ISO default; everything ingested here stays
local to your project until you (or a maintainer) deliberately fold
it into the shared default -- see "Next steps".

In [ ]:
# Mix depths and countries freely
SCOPE = [
    'CO-AN',
    'DE-BW-FB',
    'FR-IF-PA',
    'GB-EN-CO',
    'US-FL-LK',
    'US-MA-SU',
    'US-NC-CE',
    'US-TX-RR',
    'US-TX-FR',
]

# Check whether your scope has a better source

Country-specific recipes are scoped to a whole admin1
(country or territory), even when you only care about part of it.

Group your `SCOPE` entries by the admin1 they fall under,
then check each one with `find_admin_recipe_id`.

In [ ]:
from collections import defaultdict

from openplaces.core.schema import AdminId
from openplaces.recipe import find_admin_recipe_id

scope_by_admin1 = defaultdict(list)
for scope_id in SCOPE:
    admin1 = str(AdminId(scope_id).truncate_to_level(1))
    scope_by_admin1[admin1].append(scope_id)

recipe_ids_by_admin1 = {}
for admin1, scope_ids in scope_by_admin1.items():
    recipe_ids = []
    for level in [2, 3, 4]:
        recipe_id = find_admin_recipe_id(admin1, level, silent=True)
        if recipe_id:
            recipe_ids.append(recipe_id.removesuffix('.yaml'))
        found = recipe_id or '(none -- falls back to GADM)'
        print(f'{admin1} admin{level}: {found}')
    recipe_ids_by_admin1[admin1] = recipe_ids
recipe_ids_by_admin1

# Ingest it

For each admin1 with at least one recipe, ingest it -- restricted to
just your `SCOPE` entries under it, unless the whole country is in
scope. This downloads and processes only what your scope needs (e.g.
one state's worth of county subdivisions, not all fifty), and writes
it locally; it does not touch the shared `admin-spine-2026` reference
that `get_admin()` reads from by default.

In [ ]:
from openplaces.io.ingester import Ingester

for admin1, recipe_ids in recipe_ids_by_admin1.items():
    scope_ids = scope_by_admin1[admin1]
    # No restriction once the whole country is in scope -- otherwise,
    # only ingest the admin_ids you actually asked for.
    admin_ids = None if admin1 in scope_ids else scope_ids
    for recipe_id in recipe_ids:
        Ingester(recipe_id, admin_ids, verbose=True).ingest()

# Explore your scope

Since nothing was registered as the project-wide default, pass
`recipe=` explicitly to see what you just ingested -- plain
`get_admin()` (no `recipe=`) would still show GADM here.

One map, covering every `SCOPE` entry at once, however scattered
around the world -- not one map per entry, which would get unwieldy
fast for a large scope.

In [ ]:
import pandas as pd
from IPython.display import display

if SCOPE:
    frames = []
    for scope_id in SCOPE:
        admin1 = str(AdminId(scope_id).truncate_to_level(1))
        level_to_recipe_id = {
            int(r.rsplit('admin', 1)[-1]): r for r in recipe_ids_by_admin1[admin1]
        }
        own_level = AdminId(scope_id).get_level()
        finest_level = max([own_level, *level_to_recipe_id.keys()])
        admin = get_admin(
            scope_id,
            level=finest_level,
            geom=True,
            recipe=level_to_recipe_id.get(finest_level),
        )
        print(f'{len(admin):,} admin{finest_level} units for {scope_id}')
        frames.append(admin)

    scope_admin = pd.concat(frames, sort=False)
    display(scope_admin.explore(tooltip='name'))
else:
    admin = get_admin(level=1, geom=True)
    print(f'{len(admin):,} admin1 units worldwide (global default only)')
    display(admin.explore(name='world', tooltip='name'))

# Next steps

- **No recipe exists yet** for a country you care about? See
  `docs/5_contribute/writing-recipes.rst` for how recipes work, and
  `plans/lexical-churning-dragon.md` for a worked example (the GB/DE/FR
  recipes were built this way).
- **Want this to become everyone's default**, not just available
  locally for your own scope? That's a bigger, more deliberate step --
  a maintainer can fold an ingested recipe into `admin-spine-2026` with
  `openplaces.io.admin.update_admin_spine(level, recipe_id)`, which
  replaces the recipe's whole country prefix. Not something to run
  automatically as part of setup.
- Want the actual GADM boundary geometries (not just the ID reference),
  or curious how the global default itself was built?
  `05_ingest_global_administrative_units.ipynb` covers both -- you
  don't need it just to use `admin_id`s, only to draw maps of GADM's
  own boundaries.
- Need tiling data too (OpenBuildingMap, Landsat/Sentinel, census block
  groups/tracts)? See `03_ingest_tiles.ipynb`.